# LogicVol Experiment A: Regression Tsetlin Machine vs volatility baselines

This notebook runs the frozen experiment in `experiments/logicvol-v0.1.json` on the **development period only** (2008 to 2020). It does not request or store an API token, and it never sees prices after 2020-12-31.

Before starting:

1. Open **Tools → Command palette**, search for **Change runtime version** and select **26.07** (Python 3.12). The default Python 3.13 runtime cannot install the pinned `tmu==0.8.3` wheel.
2. A CPU runtime is enough. The Regression Tsetlin Machine trains in about 1 to 3 seconds per fit on CPU; the full run is roughly an hour.

On your own machine, create the development-only price file first:

```
logic-alpha logicvol-cut --csv <your Tiingo wide CSV from 2002> --output data/raw/logicvol-dev-prices.csv
```

In [ ]:
import sys
print('Python:', sys.version)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Colab runtime version 26.07 (Python 3.12), reconnect, and run again.')

## 1. Clone the public code and install pinned dependencies

Only public source code is cloned. The licensed market data are uploaded separately in the next step.

In [ ]:
from pathlib import Path

if not Path('/content/logic-alpha-tm').exists():
    !git clone https://github.com/naibwedi/logic-alpha-tm.git /content/logic-alpha-tm
%cd /content/logic-alpha-tm
!git log -1 --format='code at %h %ad: %s' --date=short
!python -m pip install -q -e ".[tm]"

## 2. Upload the development-only price file

Select `data/raw/logicvol-dev-prices.csv` from your machine. It must start in 2002 (the first test year needs about three years of history before it) and end on 2020-12-31. It stays in this temporary runtime and disappears when the runtime is deleted.

In [ ]:
import hashlib
import pandas as pd
from google.colab import files

uploaded = files.upload()
if 'logicvol-dev-prices.csv' not in uploaded:
    raise ValueError(f'Expected logicvol-dev-prices.csv, got {sorted(uploaded)}')
raw = Path('data/raw'); raw.mkdir(parents=True, exist_ok=True)
path = raw / 'logicvol-dev-prices.csv'
path.write_bytes(uploaded['logicvol-dev-prices.csv'])
del uploaded

dates = pd.read_csv(path, usecols=['date'])['date']
print('rows', len(dates), '|', dates.min(), 'to', dates.max(), '| sha256', hashlib.sha256(path.read_bytes()).hexdigest())
if dates.max() > '2020-12-31':
    raise ValueError('This file contains holdout prices. Recreate it with logic-alpha logicvol-cut.')
if dates.min() > '2003-01-31':
    raise ValueError('History starts too late; the experiment expects data from 2002.')

## 3. Optional: keep results on Google Drive

Colab can disconnect during a long run. If you save to Drive, re-running step 4 with `--resume` continues from the last finished fold instead of starting over. Skip this cell to keep everything in the temporary runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = '/content/drive/MyDrive/logicvol-v0.1'
print('Results will be written to', OUTPUT)

## 4. Run the frozen experiment

Run the smoke test first (about a minute). Then run the full experiment. If it stops part-way, run the full command again unchanged: `--resume` reuses every completed fold, but only when the code, data and spec are identical.

In [ ]:
try:
    OUTPUT
except NameError:
    OUTPUT = 'results/logicvol-v0.1'

!python -m logic_alpha_tm.cli logicvol --csv data/raw/logicvol-dev-prices.csv --output /content/logicvol-smoke --quick --assets SPY

In [ ]:
!python -m logic_alpha_tm.cli logicvol --csv data/raw/logicvol-dev-prices.csv --output "{OUTPUT}" --resume

## 5. Read the result

The decision is made on SPY by the rule frozen in the spec. QQQ, IWM and TLT are reported for robustness but do not decide pass or fail.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path(OUTPUT, 'REPORT.md').read_text()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pred = pd.read_csv(Path(OUTPUT, 'predictions-SPY.csv'), parse_dates=['date']).set_index('date')
tm_cols = [c for c in pred.columns if c.startswith('tm_seed') and not c.endswith('_var')]
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(pred.index, np.exp(pred['y']), lw=0.7, color='0.55', label='realised (next 5 days)')
ax.plot(pred.index, np.exp(pred['har_style']), lw=1.1, label='HAR-style forecast')
if tm_cols:
    ax.plot(pred.index, np.exp(pred[tm_cols].median(axis=1)), lw=1.1, label='Tsetlin Machine (median of seeds)')
ax.set_ylabel('annualised volatility'); ax.set_title('SPY next-week volatility, walk-forward test folds 2008-2020')
ax.legend(); plt.show()

## 6. Download and clean up

The ZIP contains forecasts derived from licensed prices: keep it private. After it downloads, choose **Runtime → Disconnect and delete runtime**.

In [ ]:
import shutil
archive = shutil.make_archive('/content/logicvol-v0.1', 'zip', OUTPUT)
files.download(archive)